In [2]:
!pip install openmeteo-requests
!pip install requests-cache retry-requests numpy pandas

In [3]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 52.52,
	"longitude": 13.41,
	"hourly": ["temperature_2m", "dew_point_2m", "surface_pressure", "cloud_cover", "precipitation", "precipitation_probability", "showers", "snowfall", "snow_depth", "wind_gusts_10m", "evapotranspiration", "soil_temperature_0cm", "soil_temperature_6cm", "soil_temperature_18cm", "soil_moisture_3_to_9cm", "soil_moisture_9_to_27cm", "soil_moisture_0_to_1cm", "soil_moisture_1_to_3cm"],
	"current": "precipitation",
	"past_days": 92,
	"forecast_days": 1,
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process current data. The order of variables needs to be the same as requested.
current = response.Current()
current_precipitation = current.Variables(0).Value()

print(f"\nCurrent time: {current.Time()}")
print(f"Current precipitation: {current_precipitation}")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_dew_point_2m = hourly.Variables(1).ValuesAsNumpy()
hourly_surface_pressure = hourly.Variables(2).ValuesAsNumpy()
hourly_cloud_cover = hourly.Variables(3).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(4).ValuesAsNumpy()
hourly_precipitation_probability = hourly.Variables(5).ValuesAsNumpy()
hourly_showers = hourly.Variables(6).ValuesAsNumpy()
hourly_snowfall = hourly.Variables(7).ValuesAsNumpy()
hourly_snow_depth = hourly.Variables(8).ValuesAsNumpy()
hourly_wind_gusts_10m = hourly.Variables(9).ValuesAsNumpy()
hourly_evapotranspiration = hourly.Variables(10).ValuesAsNumpy()
hourly_soil_temperature_0cm = hourly.Variables(11).ValuesAsNumpy()
hourly_soil_temperature_6cm = hourly.Variables(12).ValuesAsNumpy()
hourly_soil_temperature_18cm = hourly.Variables(13).ValuesAsNumpy()
hourly_soil_moisture_3_to_9cm = hourly.Variables(14).ValuesAsNumpy()
hourly_soil_moisture_9_to_27cm = hourly.Variables(15).ValuesAsNumpy()
hourly_soil_moisture_0_to_1cm = hourly.Variables(16).ValuesAsNumpy()
hourly_soil_moisture_1_to_3cm = hourly.Variables(17).ValuesAsNumpy()

hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	)
}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["dew_point_2m"] = hourly_dew_point_2m
hourly_data["surface_pressure"] = hourly_surface_pressure
hourly_data["cloud_cover"] = hourly_cloud_cover
hourly_data["precipitation"] = hourly_precipitation
hourly_data["precipitation_probability"] = hourly_precipitation_probability
hourly_data["showers"] = hourly_showers
hourly_data["snowfall"] = hourly_snowfall
hourly_data["snow_depth"] = hourly_snow_depth
hourly_data["wind_gusts_10m"] = hourly_wind_gusts_10m
hourly_data["evapotranspiration"] = hourly_evapotranspiration
hourly_data["soil_temperature_0cm"] = hourly_soil_temperature_0cm
hourly_data["soil_temperature_6cm"] = hourly_soil_temperature_6cm
hourly_data["soil_temperature_18cm"] = hourly_soil_temperature_18cm
hourly_data["soil_moisture_3_to_9cm"] = hourly_soil_moisture_3_to_9cm
hourly_data["soil_moisture_9_to_27cm"] = hourly_soil_moisture_9_to_27cm
hourly_data["soil_moisture_0_to_1cm"] = hourly_soil_moisture_0_to_1cm
hourly_data["soil_moisture_1_to_3cm"] = hourly_soil_moisture_1_to_3cm

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)


Coordinates: 52.52000045776367°N 13.419998168945312°E
Elevation: 38.0 m asl
Timezone difference to GMT+0: 0s

Current time: 1784369700
Current precipitation: 0.0

Hourly data
                           date  temperature_2m  dew_point_2m  \
0    2026-04-17 00:00:00+00:00             NaN           NaN   
1    2026-04-17 01:00:00+00:00             NaN           NaN   
2    2026-04-17 02:00:00+00:00             NaN           NaN   
3    2026-04-17 03:00:00+00:00             NaN           NaN   
4    2026-04-17 04:00:00+00:00             NaN           NaN   
...                        ...             ...           ...   
2227 2026-07-18 19:00:00+00:00       21.595499     10.425263   
2228 2026-07-18 20:00:00+00:00       20.795500     10.289222   
2229 2026-07-18 21:00:00+00:00       19.945499      9.794917   
2230 2026-07-18 22:00:00+00:00       18.895500      9.926531   
2231 2026-07-18 23:00:00+00:00       17.995501      9.608839   

      surface_pressure  cloud_cover  precipitation  pre

In [4]:
df = hourly_dataframe.copy()
df.head()

,date,temperature_2m,dew_point_2m,surface_pressure,cloud_cover,precipitation,precipitation_probability,showers,snowfall,snow_depth,wind_gusts_10m,evapotranspiration,soil_temperature_0cm,soil_temperature_6cm,soil_temperature_18cm,soil_moisture_3_to_9cm,soil_moisture_9_to_27cm,soil_moisture_0_to_1cm,soil_moisture_1_to_3cm
0,2026-04-17 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-04-17 01:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-04-17 02:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-04-17 03:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-04-17 04:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
df.describe()

,temperature_2m,dew_point_2m,surface_pressure,cloud_cover,precipitation,precipitation_probability,showers,snowfall,snow_depth,wind_gusts_10m,evapotranspiration,soil_temperature_0cm,soil_temperature_6cm,soil_temperature_18cm,soil_moisture_3_to_9cm,soil_moisture_9_to_27cm,soil_moisture_0_to_1cm,soil_moisture_1_to_3cm
count,1752.000000,1752.000000,1752.000000,1752.000000,1752.000000,1563.000000,1752.000000,1752.0,1752.0,1752.000000,2232.0,1752.000000,1563.000000,1563.000000,1563.000000,1563.000000,1563.000000,1563.000000
mean,18.998730,10.393873,1012.030518,62.459476,0.057477,6.673705,0.009132,0.0,0.0,23.733288,0.0,21.938768,22.838659,22.004940,0.273434,0.280011,0.271808,0.272210
std,6.148754,4.093050,6.865180,39.503059,0.784073,17.778687,0.080596,0.0,0.0,11.001148,0.0,8.554059,6.352226,4.691365,0.018237,0.017669,0.018054,0.018123
min,4.169500,-0.901978,991.855774,0.000000,0.000000,0.000000,0.000000,0.0,0.0,1.800000,0.0,3.319500,5.919500,9.369500,0.237000,0.249000,0.234000,0.234000
25%,14.983000,7.497406,1008.648743,22.000000,0.000000,0.000000,0.000000,0.0,0.0,15.480000,0.0,16.095499,18.445499,19.445499,0.257000,0.263000,0.255000,0.256000
50%,18.995501,10.438704,1012.890259,79.000000,0.000000,0.000000,0.000000,0.0,0.0,21.959999,0.0,20.995501,22.295500,22.245501,0.279000,0.283000,0.277000,0.278000
75%,22.807999,13.003731,1016.159485,100.000000,0.000000,0.000000,0.000000,0.0,0.0,30.960001,0.0,27.570499,27.120499,24.995501,0.286000,0.291000,0.284000,0.284000
max,39.495502,21.266012,1028.722046,100.000000,22.900000,100.000000,2.100000,0.0,0.0,61.560001,0.0,48.895500,43.245502,34.995502,0.334000,0.344000,0.322000,0.326000


In [6]:
print('📋 Basic dataset info displayed')
info_df = pd.DataFrame({
    'Data Type': df.dtypes,
    'Non-null': df.notnull().sum(),
    'Missing Values': df.isnull().sum(),
    'Unique Values': df.nunique()
})

info_df

📋 Basic dataset info displayed


,Data Type,Non-null,Missing Values,Unique Values
date,"datetime64[s, UTC]",2232,0,2232
temperature_2m,float32,1752,480,684
dew_point_2m,float32,1752,480,1591
surface_pressure,float32,1752,480,1730
cloud_cover,float32,1752,480,101
precipitation,float32,1752,480,22
precipitation_probability,float32,1563,669,41
showers,float32,1752,480,11
snowfall,float32,1752,480,1
snow_depth,float32,1752,480,1


In [7]:
df_clean = df.dropna().reset_index(drop=True)
print(f"Removed {len(df) - len(df_clean)} null rows")
df_clean.head()

Removed 669 null rows


,date,temperature_2m,dew_point_2m,surface_pressure,cloud_cover,precipitation,precipitation_probability,showers,snowfall,snow_depth,wind_gusts_10m,evapotranspiration,soil_temperature_0cm,soil_temperature_6cm,soil_temperature_18cm,soil_moisture_3_to_9cm,soil_moisture_9_to_27cm,soil_moisture_0_to_1cm,soil_moisture_1_to_3cm
0,2026-05-14 21:00:00+00:00,8.969500,6.410766,994.017395,0.0,0.0,0.0,0.0,0.0,0.0,6.12,0.0,7.1695,9.7695,11.3695,0.334,0.344,0.322,0.326
1,2026-05-14 22:00:00+00:00,8.569500,6.190332,994.807251,0.0,0.0,0.0,0.0,0.0,0.0,4.68,0.0,6.4195,9.0695,11.2195,0.331,0.343,0.320,0.323
2,2026-05-14 23:00:00+00:00,8.169499,5.967039,994.999695,1.0,0.0,0.0,0.0,0.0,0.0,3.60,0.0,5.8695,8.4695,11.0195,0.329,0.342,0.318,0.321
3,2026-05-15 00:00:00+00:00,6.519500,5.468820,995.470581,18.0,0.0,0.0,0.0,0.0,0.0,3.60,0.0,4.2195,7.2695,10.7695,0.327,0.341,0.316,0.320
4,2026-05-15 01:00:00+00:00,6.219500,5.325260,995.266602,0.0,0.0,0.0,0.0,0.0,0.0,2.88,0.0,3.7695,6.8195,10.5195,0.325,0.339,0.315,0.318
